# Debugging AlexNet model and training

## Network basics

In [ ]:
from src.models import lipsalexnet
import torch
import importlib

In [ ]:
importlib.reload(lipsalexnet)

In [ ]:
config = {
    "hparams": {
        "num_classes": 1000,
        "lr": 0.001,
        "w_max": 2.0,
        "projection": "spectral_normalize",
    },
    "optim_settings": {
        "optimizer": "Adam",
        "lr": 0.001,
        "weight_decay": 0.0001,
        "betas": (0.9, 0.999),
        "eps": 1e-8,
    },
}

In [ ]:
test_model = lipsalexnet.LipsAlexNetModule(config)

In [ ]:
fake_input = torch.randn(1, 3, 224, 224)
test_model(fake_input).shape

## Training code

In [1]:
import json
import os
import pathlib
import socket
# from argparse import ArgumentParser

import torch
import yaml
from lightning import Trainer, seed_everything
from lightning.pytorch.callbacks import ModelCheckpoint

from src.models.lipsvision import get_module


torch.set_float32_matmul_precision("medium")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [2]:
hostname = socket.gethostname()

In [3]:
seed = 42
config = "configs/lipsalexnet_test.json"
num_workers = 2
gpus = 1
exp_dir = pathlib.Path("experiments")
resume_training = True
arg_ckpt_path = ""
num_nodes = 1

In [4]:
seed_everything(seed)

[rank: 0] Seed set to 42


42

In [5]:
config_path = config
print(f"Loading config from {config_path}")

if config_path.endswith(".json"):
    with open(config_path, "r") as f:
        config = json.load(f)
else:
    with open(config_path, "r") as f:
        config = yaml.load(f, Loader=yaml.FullLoader)

Loading config from configs/lipsalexnet_test.json


In [6]:
module = get_module(config)
print(f"Module: {module}")

Module: <class 'src.models.lipsalexnet.LipsAlexNetModule'>


In [7]:
config["num_workers"] = num_workers
config["ngpus"] = gpus

In [8]:
config_path = pathlib.Path(config_path)
checkpoint_dir = exp_dir / f"{config_path.stem}/checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
ckpt_paths = sorted(checkpoint_dir.glob("*.ckpt"), key=os.path.getctime)

In [9]:
ckpt_path = None
if resume_training and (len(ckpt_paths) > 0 or arg_ckpt_path != ""):
    if arg_ckpt_path != "":
        ckpt_path = arg_ckpt_path
        model = module.load_from_checkpoint(ckpt_path, config=config)
    else:
        ckpt_path = ckpt_paths[-1]
        model = module.load_from_checkpoint(ckpt_path, config=config)
else:
    model = module(config)

In [10]:
trainer = Trainer(
    precision="32",
    default_root_dir=exp_dir / config_path.stem,
    max_epochs=config["hparams"]["max_epochs"],
    num_nodes=num_nodes,
    devices=gpus,
    accelerator="gpu",
    limit_val_batches=config["hparams"].get("limit_val_batches", 1.0),
    limit_train_batches=config["hparams"].get("limit_train_batches", 1.0),
    val_check_interval=config["hparams"].get("val_check_interval", 1.0),
    gradient_clip_val=config["hparams"].get("gradient_clip_val", None),
    gradient_clip_algorithm=config["hparams"].get("gradient_clip_algorithm", "value"),
    accumulate_grad_batches=config["hparams"].get("accumulate_grad_batches", 1),
    profiler=config["hparams"].get("profiler", None),
    callbacks=[
        ModelCheckpoint(
            monitor="val_loss",
            mode="min",
            save_top_k=1,
            save_last=True,
            save_weights_only=True,
        )
    ],
)

/home/rphess/conda/envs/new_metam_env/lib/python3.9/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/rphess/conda/envs/new_metam_env/lib/python3.9/ ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/rphess/conda/envs/new_metam_env/lib/python3.9/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
`Trainer(limit_t

In [11]:
trainer.fit(model)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name           | Type              | Params | Mode 
-------------------------------------------------------------
0 | model          | Sequential        | 2.5 M  | train
1 | fake_relu_dict | ModuleDict        | 0      | train
2 | avgpool        | AdaptiveAvgPool2d | 0      | train
3 | classifier     | Sequential        | 58.6 M | train
-------------------------------------------------------------
61.1 M    Trainable params
0         Non-trainable params
61.1 M    Total params
244.367   Total estimated model params size (MB)
31        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/rphess/conda/envs/new_metam_env/lib/python3.9/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Training: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


AttributeError: 'tuple' object has no attribute 'tb_frame'